   - Basic concurrency TERMINOLOGY
   - Disabling Interrupts
   - Semaphores
   - Locks
   - Monitors

The PintOS SYNCHRONISATION PRIMITIVES ARE IMPLEMENTED IN: 
   `src/threads/synch.ch`

---
Why do we need to care about concurrency?
   - ... execution of a thread can be interupted at any time
   - Incomplete/unfinished tasks may harm the SAFETY / CORRECTNESS of other
     tasks, or EVEN THE WHOLE OS. 
   - CONCURRENT THREADS are often NON-DETERMINSTIC
      (Within one )
   - The CORRECTNESS of our OS shouldn't depend on the RELATIVE TIMING OF EVENTS
     in concurrent computations. 




THREAD
   - An EXECUTION CONTEXT, that is, all of the information that the CPU needs to
     be able to execute a stream of instructions

ATOMIC
   - A UNITARY ACTION that is essentially INDIVISIBLE, IRREDUCIBLE and TAKES
     EFFECT IN ONE STEP.

RACE CONDITION
   - When two or more threads access shared data at the same time, at least
     one of them is modifing it non-atomically and this results in undefined or
     incorrect behaviour

DEADLOCK 
   - A situation in which TWO OR MORE THREADS sharing the same resources are
     effectively preventing each other from accessing those resources resulting
     in all of those threads failing to make progress indefinitely.


---
   INTERNAL INTERRUPTS (cannot be ignored)
   EXTERNAL INTERRUPTS (can be postponed)

INTERRUPT HANDLER
   A kernel routine that processes (handles) a particular interrupt signal with
   exclusive access to the CPU. 

---
   - PREVENTS the CPU from responding to EXTERNAL INTERRUPTS (i.e. the running
     thread cannot be pre-empted)
     All operations are then effectivel ATOMIC. 

`src/threads/synch.ch`

- `unsigned value`
   * WHAT IT REPRESENTS: The number of available permits or resource units
   * WHY IT IS NEEDED: It tells incoming threads whether they can proceed
     immediately (`value > 0`) or must wait (`value == 0`)

---

   A MONITOR is not a low-level hardware primitive; it is an architectural
   pattern that bundles three things together into a single, cohesive package:
   
   1. SHARED DATA VARIABLES: The state you want to protect (e.g. `int count = 0;`)
   2. A MONITOR LOCK: A mutex that guarantees only ONE THREAD can be inside the
      monitor execut9ing code at any given time
   3. CONDITION VARIABLES: Dedicated "waiting lounges" where a thread can
      temporarily sleep if the shared data isn't in the state it needs ... 

---

   In PintOS, the Intel 8254 Programmable Interval Timer is a hardware chip used
   to drive the system clock by generating periodic interrupts at a fixed
   frequency ... Every time the PID fires a hardware interrupt (specifically 
   `IRQ 0`), the CPU interrupts the currently running thread and invokes the 
   Pintos timer interrupt handler. This mechanism provides the foundational
   heartbeat for the OS, enabling curcual kernel operations such as tracking
   elapsed time, implementing thread sleeping functions (`timer_sleep`), 
   managing thead scheduling and triggering time-slice context switches to
   ensure fair CPU utilization among competing threads. 

   The term "PERIODIC INTERRUPT" is used precisely because it does exactly what
   it says ... forces the CPU to stop executing the current thread 100 times 
   per second to let the kernel run ... While this might seem counterproductive
   at first glance, it is actually the foundational design that makes modern,
   responsive operating systems possible. 

   Here's wh this mechanism is essential rather than inefficient:
   
   - PREVENTING CPU MONOPOLISATION: Without these periodic interrupts, a single
     thread with an infinite loop (like `while(true) {}`) would lock up your
     entire computer forever. The operating system would lose control becasue it 
     has no way to force its way back into the CPU. The PID heartbeat acts as a 
     guaranteed fallback mechanism to ensure the kernel always regains control.
   - IMPLEMENTING FAIR SCHEDULING: To create the illusion that multiple programs
     are running at the same time (multitasking), the OS must rapidly swap
     threads in and out. The timer interrupt acts as a referee, checking every
     10 milliseconds if the current thread has used up its "time slice" so
     another waiting thread can get a turn 
   - NEGLIGIBLE PERFORMANCE COST: Modern CPU operate at billions fo cycles per 
     second ... pausing a thread 100 times a second to run a highly optimized
     kernel function takes only a fraction of a microsecond. The CPU overhead is
     well under 1%, making the performance trade-off practically unnoticeable
     to the user while gaining complete system stability and multitaskng 
     capabilities. 

---

   A KERNEL is the core of an operating system. It is the one program that runs 
   with full control of the hardware. It decides which thread runs, handles 
   interrupts, manages memory, and talks to devices. In Pintos, the kernel is 
   the whole operating ssytem you build: the code in:
   - src/threads
   - src/devices
   - src/lib
   - src/userprog
   - src/vm
   Normal user programs only arrive in ... Until then, everything you run...
   runs inside the kernel..

   `kernel.o` comes from the build ... The compiler turns each `.c` file into 
   its own `.o` file: machine code plus debug information, but not yet a 
   runnable program. The linker then joins all of these into one file ...
   `kernel.o` which holds the whole kernel. it keeps the debug information,
   which maps each machine address back to a function, file and line. That is why
   the debugging tools use it. `pintos-gdb kernel.o` debugs with it, and the
   `backtrace` tool uses it to panic addresses into lines ...

   `kernel.bin` is the same machine code with the debug information removed. It
   is the exact bytes that go into memory 

-- `src/userprog` dir does not contain the actual applications a user writes;
   instead, it contains the KERNEL'S INTERNAL CODE responsible for managing
   those applications--such as the system call handler (`syscall.c`), the 
   executable file loader, and process memory initilisation.

   It is compiled directly into the OS proper because it acts as the privileged
   bridge that allows isolated user programs to safely run. 

-- mental image spot on ... the kernel is exactly the core software running with
   hardwre privilege that sits directly between normal applications and the 
   physical machine. For example, the macOS kernel (called XNU) literally 
   includes the low-tech Mach code and I/O kit frameworks that initialises your
   Apple Silicon CPU, configure its memory management unit (MMU), and handle
   hardware interrupts, while standard user applications simply live in an 
   unprivileged space and must ask that kernel for permission to do anything ... 

---

   ... Pinto uses two sentinels... `head` and `tail`, in a straight line ...
   The `prev` of ... are `NULL`. Your Set A list used one sentinel in a circle:
   its `next` was the first element and is `prev` was the last. The comment
   admits that ... single-header element design would've worked just as well.
   Both designs remove the special cases. Every real element always has a real
   neighbour on each side, so `list_remove` needs two pointer writes and no 
   `if`. The loop idiom also looks the same. Yours stopped at `e != &head`,
   and Pintos stopped at `e != list_end (&list)`, which returns the
   address of `tail`.

   The reason for two is the last sentence: "a little bit of checking". In
   Pintos, a sentinel ha a different shape from an element. A header has
   exactly one `NULL` link and an interior element has none. So `list.c` can
   test whether a pointer is a header or a real element, and `ASSERT` on that,
   for example when you tru to remove a header or step past the end. Your
   circular sentinel has the same shape as an element, so nothing could tell 
   them apart. This is why `CONTAINER_OF (&head)` bug ran silently until
   AddressSanitizer caught it. Pintos pays one extra `struct list_elem` per list
   to catch that class of bug with a kernel panic at the exact point of the 
   mistake. 
   

   When creating a fresh thread in `thread_create()`, the kernel thread-frame
   is manually manufactured from scratch to act as a forgery.

   Because a brand-new thread has never run before, it doesn't have a natural 
   CPU state to save. Instead, `thread_create()` intentionally formats the top
   of the new thread's stack to MIMIC a thread that was previously suspended
   by `switch_threads()`

   When the scheduler eventually picks up this new thread for the first time,
   `switch_threads()` will cleanly read this forged frame and trick the CPU into
   executing the thread's entry function. 

   ...

   Directly below that, space is allocated for a `struct switch_threads_frame`. 
   This is the critical structure that `switch_threads()` expects to pop off the
   stack during a context switch. 
   - The registers `ebp` ... `esi` are left uninitialized (or set to 0) because
     their starting values do not matter for a new thread.
   - THE MAGGIC HAPPENS AT THE `eip` (instruction pointer): The kernel manually
     writes the memory address of the assembly function
